In [1]:
import pandas as pd 


Nov = pd.read_csv('data/Divvy Data/202511-divvy-tripdata.csv')
Dec = pd.read_csv('data/Divvy Data/202512-divvy-tripdata.csv')
Jan = pd.read_csv('data/Divvy Data/202601-divvy-tripdata.csv')
Feb = pd.read_csv('data/Divvy Data/202602-divvy-tripdata.csv')
Mar = pd.read_csv('data/Divvy Data/202603-divvy-tripdata.csv')
Apr = pd.read_csv('data/Divvy Data/202604-divvy-tripdata.csv')
May = pd.read_csv('data/Divvy Data/202605-divvy-tripdata.csv')
Jun = pd.read_csv('data/Divvy Data/202606-divvy-tripdata.csv')
Jul = pd.read_csv('data/Divvy Data/202607-divvy-tripdata.csv')
Aug = pd.read_csv('data/Divvy Data/202608-divvy-tripdata.csv')
Sep = pd.read_csv('data/Divvy Data/202609-divvy-tripdata.csv')
Oct = pd.read_csv('data/Divvy Data/202610-divvy-tripdata.csv')





In [2]:
rides = pd.concat(
    [Nov, Dec, Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct], 
ignore_index = True) 


rides['started_at'] = pd.to_datetime(rides['started_at'])
# rides['started_at'] = rides['started_at'].dt.tz_localize('America/Chicago')
rides['ended_at'] = pd.to_datetime(rides['ended_at'])
# rides['ended_at'] = rides['ended_at'].dt.tz_localize('America/Chicago')

In [3]:
# Cubs
# https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.mlb.com/cubs/schedule/downloadable-schedule&ved=2ahUKEwiFw6Gz8JKYAxWIN4YAHQUQGPoQFnoECB8QAQ&usg=AOvVaw0pq6KeoTn26KI0j57L_tHs

#Bulls, Bears
#fixturedownload.com

In [4]:

B1 = pd.read_csv('data/bears1.csv')
B2 = pd.read_csv('data/bears2.csv')

NFL = pd.concat([B1, B2], ignore_index=True)


Home = (NFL['Location'] == 'Soldier Field') 
Bears = NFL[Home].copy()

Bears['Date'] = pd.to_datetime(Bears['Date'], format='%d/%m/%Y %H:%M', utc=True)     # AI helped w this part
Bears['Date'] = Bears['Date'].dt.tz_convert('America/Chicago')

start = pd.Timestamp("2025-11-01", tz="America/Chicago")
end = pd.Timestamp("2026-11-01", tz="America/Chicago")   

Bears = Bears[(Bears['Date'] >= start) & (Bears['Date'] < end)]
print(len(Bears))


11


In [5]:
NBA = pd.read_csv('data/Bulls.csv')
Home2 = (NBA['Location'] =='United Center')
Bulls = NBA[Home2].copy()

Bulls['Date'] = pd.to_datetime(Bulls['Date'], format = '%d/%m/%Y %H:%M', utc = True)
Bulls['Date'] = Bulls['Date'].dt.tz_convert('America/Chicago')

Bulls = Bulls[(Bulls['Date'] >= start) & (Bulls['Date'] < end)]
print(len(Bulls))

37


In [6]:
Cubs = pd.read_csv('data/Cubs.csv')

Cubs['Date'] = pd.to_datetime(
    Cubs['START DATE'] + ' ' + Cubs['START TIME'],
    format='%m/%d/%y %I:%M %p'
)
Cubs['Date'] = Cubs['Date'].dt.tz_localize('America/Chicago')

Cubs = Cubs[(Cubs['Date'] >= start) & (Cubs['Date'] < end)]
print(len(Cubs))

81


In [7]:
Wrigley = [41.9484 , -87.6558]
Soldier = [41.8623, -87.6167]
United = [41.880692, -87.674370]


Checklist 
- clean divvy data (missing vals [drop or impute], duplicates, outliers)
- Build one table of unique stations with a single lat/lng each (station coordinates can wobble, so use the median per ID)
- Compute each station’s distance to each venue (haversine) and choose a radius, like 0.5 mile, for “nearby”
- Compute the baseline from non-event days only, matched on month and weekday vs. weekend. If you average the whole month, the event days contaminate the baseline
- Compute Net flow (arrivals minus departures)
- Compute the lift (event-day departures minus baseline, and the percent change) per station and per team
- plot stuff






In [8]:

missing_percent = rides.isna().mean() * 100
missing_percent.sort_values(ascending=False).round(1)


end_station_name      21.7
end_station_id        21.7
start_station_name    20.7
start_station_id      20.7
end_lat                0.1
end_lng                0.1
ride_id                0.0
rideable_type          0.0
started_at             0.0
ended_at               0.0
start_lat              0.0
start_lng              0.0
member_casual          0.0
dtype: float64

In [9]:
print(rides.duplicated(subset = ['ride_id']).sum())
print(rides.duplicated().sum())
rides = rides.drop_duplicates()

35
35


In [10]:
print(rides['started_at'].min())
print(rides['started_at'].max())

rides['month'] = rides['started_at'].dt.month


print(rides['start_lat'].max())
print(rides['start_lat'].min())
print(rides['start_lng'].max())
print(rides['start_lng'].min())

print(rides['end_lat'].max())
print(rides['end_lat'].min())
print(rides['end_lng'].max())
print(rides['end_lng'].min())




2025-10-31 01:24:38.549000
2026-10-31 23:59:45.105000
42.07
41.64
-87.52
-87.91
42.19
41.51
-87.37
-88.1


In [11]:
valid = (rides['ended_at'] > rides['started_at']) 

rides = rides[valid].copy()
rides['duration'] = rides['ended_at'] - rides['started_at']
rides['duration_min'] = rides['duration'].dt.total_seconds() / 60


print(rides['duration_min'].describe())
too_short = rides['duration_min'] < 1
too_long = rides['duration_min'] > 240 

print(len(rides[too_long]) / len(rides) * 100)
print(len(rides[too_short]) / len(rides) * 100)

keep = ~(too_short | too_long)
rides = rides[keep].copy()

print(rides['duration_min'].describe()) 


count    6.294387e+06
mean     1.506885e+01
std      4.991845e+01
min      9.333333e-04
25%      5.304467e+00
50%      9.197633e+00
75%      1.602335e+01
max      1.559950e+03
Name: duration_min, dtype: float64
0.17676065993400153
2.5362755737770812
count    6.123618e+06
mean     1.361576e+01
std      1.493139e+01
min      1.000000e+00
25%      5.562567e+00
50%      9.421958e+00
75%      1.623335e+01
max      2.398880e+02
Name: duration_min, dtype: float64


In [12]:

known_station_id = rides.dropna(subset = 'start_station_id').copy()

stations = known_station_id.groupby('start_station_id').agg(
    Station_name= ('start_station_name', 'first'),
    n_names = ('start_station_name', 'nunique'),
    median_lat = ('start_lat' , 'median'), 
    median_lng = ('start_lng', 'median'),
    total_rides = ('ride_id', 'count'), 
    first_date = ('started_at', 'min'),
    last_date = ('started_at', 'max')

)

stations = stations.reset_index()

print(len(stations) == known_station_id['start_station_id'].nunique())






True


In [13]:
stations.head(5) 






,start_station_id,Station_name,n_names,median_lat,median_lng,total_rides,first_date,last_date
0,CHI02331,Franklin St & Randolph St,1,41.884390,-87.635170,632,2026-05-15 09:14:57.472,2026-06-30 19:16:09.237
1,CHI00005,Milwaukee Ave & Wabansia Ave,1,41.912616,-87.681469,6321,2025-12-11 19:59:10.652,2026-10-31 22:20:13.835
2,CHI00213,Archer Ave & 43rd St,1,41.815491,-87.702441,11,2025-11-07 16:16:17.178,2026-03-13 18:31:30.097
3,CHI00214,Campbell Ave & Montrose Ave,1,41.961524,-87.691177,3819,2025-11-01 07:42:29.625,2026-10-31 23:55:34.814
4,CHI00215,Avondale Ave & Irving Park Rd,1,41.953393,-87.732002,2467,2025-11-01 00:23:25.161,2026-10-31 20:55:43.945


In [14]:
from haversine import Unit, haversine


         

stations['dist_SF'] = stations.apply(
    lambda row: haversine(list((row['median_lat'], row['median_lng'])) , Soldier, unit = Unit.MILES), 
    axis = 1
)



stations['dist_Wrig'] = stations.apply(
    lambda row: haversine(list((row['median_lat'], row['median_lng'])) , Wrigley, unit = Unit.MILES), 
        axis = 1
)

stations['dist_UC'] = stations.apply(
    lambda row: haversine(list((row['median_lat'], row['median_lng'])) , United, unit = Unit.MILES), 
        axis = 1
)



